# INBreast Breast Regions

Visualize breast bounding boxes from `INBreast_breast_region.csv` overlaid on the corresponding DICOM images.

In [ ]:
import ast
import csv
import random
from pathlib import Path
import sys

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from src.preprocessing import load_dicom_pixels

# INBreast dataset root directory
INBREAST_ROOT = Path.home() / 'Escritorio/Datasets/INbreast'
INBREAST_CSV = INBREAST_ROOT / 'INBreast_breast_region.csv'

In [ ]:
# Load CSV into a list of dicts
with INBREAST_CSV.open(newline='', encoding='utf-8') as f:
    records = list(csv.DictReader(f))

print(f"Total annotations: {len(records)}")
print("Sample row:", records[0])

In [ ]:
def show_dicom_with_bbox(record, inbreast_root=INBREAST_ROOT, ax=None):
    """Display a DICOM image with its breast bounding box.

    Parameters
    ----------
    record : dict
        Row from INBreast_breast_region.csv with 'file_name' and 'bboxes'.
    inbreast_root : Path, optional
        Root directory of the INBreast dataset. Defaults to INBREAST_ROOT.
    ax : matplotlib.axes.Axes, optional
        Axes to draw on. If None, a new figure is created.
    """
    # Build full path from relative path in CSV
    path = Path(inbreast_root) / record['file_name']
    # bboxes format: [x1, y1, x2, y2]
    x1, y1, x2, y2 = ast.literal_eval(record['bboxes'])
    score = float(record['scores'])

    # Load DICOM pixels (normalized)
    pixels = load_dicom_pixels(path, normalize=True)

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 8))

    ax.imshow(pixels, cmap='gray', aspect='equal')

    rect = mpatches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor='lime', facecolor='none',
        label=f'breast (score={score:.3f})'
    )
    ax.add_patch(rect)
    ax.legend(loc='upper right', fontsize=8)
    ax.set_title(path.stem, fontsize=8)
    ax.axis('off')
    return ax

In [ ]:
# Show a single random example
record = random.choice(records)
show_dicom_with_bbox(record)
plt.tight_layout()
plt.show()

In [ ]:
# Show a 2×3 grid of random examples
sample = random.sample(records, 6)
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
for ax, rec in zip(axes.flat, sample):
    show_dicom_with_bbox(rec, ax=ax)
plt.suptitle('INBreast – Breast Region Bounding Boxes', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Show images by laterality (L or R)
LATERALITY = 'R'  # change to 'L' for left breast

lat_records = [r for r in records if f'_MG_{LATERALITY}_' in r['file_name']]
print(f"Found {len(lat_records)} annotations for laterality {LATERALITY}")

if lat_records:
    # Show first 4
    sample = lat_records[:4]
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    for ax, rec in zip(axes.flat, sample):
        show_dicom_with_bbox(rec, ax=ax)
    plt.suptitle(f'INBreast – {LATERALITY} Breast', fontsize=13)
    plt.tight_layout()
    plt.show()